# 03 · 构建神经网络：nn.Module 与常用层

> 掌握 `nn.Module` 就掌握了 PyTorch 建模的骨架。
> 任何网络（CNN、RNN、Transformer）都是 `nn.Module` 的子类。

## 3.1 nn.Module 是什么

`nn.Module` 是所有神经网络模块的基类。它负责：
1. **自动注册参数**：`self.xxx = nn.Linear(...)` 后，参数会出现在 `model.parameters()` 里。
2. **管理子模块**：支持嵌套。
3. **提供 `train()` / `eval()` / `to(device)` 等工具**。

### 必须实现两个东西
```python
class MyNet(nn.Module):
    def __init__(self):
        super().__init__()          # 必须调用
        # 定义层
    def forward(self, x):
        # 定义数据怎么流动
        return ...
```

In [ ]:
import torch
import torch.nn as nn

class MyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 20)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(20, 3)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = MyNet()
x = torch.randn(4, 10)        # 4 个样本，每个 10 维
out = model(x)                # 自动调用 forward
print("输出形状:", out.shape)

# 直接调用 model(x) 实际上是 __call__ -> forward，还包含 hook 等逻辑
# 永远不要直接调用 model.forward(x)

## 3.2 参数与 nn.Parameter

`nn.Parameter` 是一种特殊的张量：
- 默认 `requires_grad=True`
- 被赋值给 Module 的属性时，会自动注册

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # 手动创建参数（等价于 Linear 内部的 weight/bias）
        self.weight = nn.Parameter(torch.randn(3, 5) * 0.01)
        self.bias = nn.Parameter(torch.zeros(3))

    def forward(self, x):
        return x @ self.weight.t() + self.bias

net = Net()
print("参数列表:")
for name, p in net.named_parameters():
    print(f"  {name:8s} shape={tuple(p.shape)} requires_grad={p.requires_grad}")

# 所有参数的总数
total = sum(p.numel() for p in net.parameters())
print("总参数量:", total)

### 三种遍历参数的方法

| 方法 | 返回 |
|---|---|
| `model.parameters()` | 只返回参数张量 |
| `model.named_parameters()` | (名字, 参数) |
| `model.state_dict()` | 有序字典（保存模型用） |
| `model.named_modules()` | (名字, 子模块) |
| `model.children()` / `modules()` | 子模块 |
| `model.buffers()` | 非参数但需保存的张量（如 BN 的 running_mean） |

In [ ]:
class Block(nn.Module):
    def __init__(self, inp, out):
        super().__init__()
        self.linear = nn.Linear(inp, out)
        self.bn = nn.BatchNorm1d(out)

    def forward(self, x):
        return self.bn(self.linear(x))

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = Block(10, 20)
        self.block2 = Block(20, 5)

    def forward(self, x):
        return self.block2(self.block1(x))

net = Net()
print("=== named_parameters ===")
for n, p in net.named_parameters():
    print(" ", n, tuple(p.shape))

print("\n=== named_modules ===")
for n, m in net.named_modules():
    print(" ", repr(n), "->", type(m).__name__)

print("\n=== state_dict 的 key ===")
for k in net.state_dict():
    print(" ", k)

## 3.3 常用层速查

### 全连接层 nn.Linear

$$y = xW^T + b$$

`nn.Linear(in_features, out_features, bias=True)`
- 输入 `(..., in_features)`，输出 `(..., out_features)`（支持任意前置维度）

In [ ]:
fc = nn.Linear(4, 3)
print("weight shape (out,in):", fc.weight.shape)
print("bias shape           :", fc.bias.shape)

x = torch.randn(2, 5, 4)     # (batch, seq, feature)
print("输入 (2,5,4) -> 输出:", fc(x).shape)   # (2,5,3)

# 手动验证
print("手动计算一致:", torch.allclose(fc(x), x @ fc.weight.t() + fc.bias, atol=1e-5))

### 激活函数

| 层 | 公式 | 特点 |
|---|---|---|
| `nn.ReLU` | max(0,x) | 最常用，简单高效 |
| `nn.LeakyReLU` | max(αx,x) | 解决 ReLU 死亡 |
| `nn.Sigmoid` | 1/(1+e⁻ˣ) | 输出 (0,1)，易梯度消失 |
| `nn.Tanh` | (eˣ-e⁻ˣ)/(eˣ+e⁻ˣ) | 输出 (-1,1) |
| `nn.Softmax` | eˣⁱ/Σeˣ | 转概率 |
| `nn.LogSoftmax` | log(softmax) | 配 NLLLoss |
| `nn.GELU` | 平滑版 ReLU | Transformer 常用 |
| `nn.SiLU`(Swish) | x·sigmoid(x) | 现代网络常用 |

In [ ]:
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
print("ReLU       :", nn.ReLU()(x))
print("LeakyReLU  :", nn.LeakyReLU(0.1)(x))
print("Sigmoid    :", nn.Sigmoid()(x))
print("Tanh       :", nn.Tanh()(x))
print("GELU       :", nn.GELU()(x))
print("SiLU       :", nn.SiLU()(x))

logits = torch.tensor([[1.0, 2.0, 3.0]])
print("\nSoftmax    :", nn.Softmax(dim=1)(logits))
print("LogSoftmax :", nn.LogSoftmax(dim=1)(logits))

### 常用小层：Dropout / BatchNorm / LayerNorm

- `nn.Dropout(p)`：训练时以概率 p 随机置零（防过拟合）；**eval 时不生效**。
- `nn.BatchNorm1d(d)`：对 batch 内每个特征做标准化。
- `nn.LayerNorm(d)`：对每个样本的特征做标准化（Transformer 用）。

In [ ]:
x = torch.ones(1, 5) * 10

drop = nn.Dropout(p=0.5)
drop.train()
print("Dropout (train):", drop(x))      # 有 0，且其余值被放大 1/(1-p)
drop.eval()
print("Dropout (eval) :", drop(x))      # 不变

bn = nn.BatchNorm1d(5)
bn.train()
y = bn(torch.randn(8, 5) * 3 + 5)
print("\nBatchNorm 后 mean≈0,std≈1:", y.mean().item(), y.std().item())

ln = nn.LayerNorm(5)
z = ln(torch.randn(8, 5) * 3 + 5)
print("LayerNorm 后每行 mean≈0,std≈1:", z.mean(dim=1).abs().max().item())

> **核心区别**：`dropout`/`batchnorm` 在训练和推理时行为不同，
> 所以必须正确调用 `model.train()` / `model.eval()`。

### 卷积层 nn.Conv2d（第七章详解）

```python
nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0)
```
输入 `(N, C_in, H, W)` → 输出 `(N, C_out, H_out, W_out)`

In [ ]:
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
x = torch.randn(1, 3, 32, 32)
print("Conv2d:", x.shape, "->", conv(x).shape)
print("weight:", conv.weight.shape, " bias:", conv.bias.shape)

pool = nn.MaxPool2d(2)
print("MaxPool2d:", conv(x).shape, "->", pool(conv(x)).shape)

gap = nn.AdaptiveAvgPool2d(1)
print("全局平均池化:", conv(x).shape, "->", gap(conv(x)).shape)

## 3.4 容器：Sequential / ModuleList / ModuleDict

### nn.Sequential：顺序堆叠（最常用）

In [ ]:
# 写法 1：直接传层
model = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)
print(model)
print("输出:", model(torch.randn(4, 784)).shape)

# 写法 2：OrderedDict，给层命名（保存参数时清晰）
from collections import OrderedDict
model2 = nn.Sequential(OrderedDict([
    ("fc1", nn.Linear(784, 128)),
    ("act", nn.ReLU()),
    ("fc2", nn.Linear(128, 10)),
]))
print("\n带名字:", list(model2.state_dict().keys()))

### nn.ModuleList / nn.ModuleDict

当层数量不固定、需要循环创建时，**必须**用 ModuleList 而不是普通 Python list，
否则参数不会被注册！

In [ ]:
class MLP(nn.Module):
    def __init__(self, dims):
        super().__init__()
        # 正确：用 ModuleList
        self.layers = nn.ModuleList([
            nn.Linear(dims[i], dims[i + 1]) for i in range(len(dims) - 1)
        ])
        self.act = nn.ReLU()

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = self.act(x)
        return x

net = MLP([10, 32, 32, 2])
print("参数数量:", len(list(net.parameters())))   # 3 层 * (w,b) = 6
print("输出:", net(torch.randn(5, 10)).shape)

# 反例：普通 list 不会注册参数
class Bad(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [nn.Linear(10, 10)]     # 错误！
    def forward(self, x):
        return self.layers[0](x)

print("\n错误写法参数数:", len(list(Bad().parameters())))   # 0

In [ ]:
# ModuleDict：给子模块起名字，方便按名字取用
class MultiHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.heads = nn.ModuleDict({
            "cls": nn.Linear(16, 10),
            "reg": nn.Linear(16, 1),
        })

    def forward(self, x, task):
        return self.heads[task](x)

m = MultiHead()
x = torch.randn(4, 16)
print("分类头:", m(x, "cls").shape, " 回归头:", m(x, "reg").shape)

## 3.5 参数初始化

默认初始化通常还行，但深层网络有时需要手动初始化。

| 方法 | 适用 |
|---|---|
| `nn.init.xavier_uniform_` | tanh/sigmoid 激活 |
| `nn.init.kaiming_normal_` | ReLU 激活（推荐） |
| `nn.init.zeros_ / ones_` | 偏置 |
| `nn.init.normal_(mean,std)` | 正态 |

In [ ]:
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
        nn.init.zeros_(m.bias)

net = nn.Sequential(nn.Linear(10, 20), nn.ReLU(), nn.Linear(20, 1))
print("初始化前 w 的均值/标准差:", net[0].weight.mean().item(), net[0].weight.std().item())

net.apply(init_weights)     # 递归应用到所有子模块
print("初始化后 w 的标准差:", net[0].weight.std().item(), "(kaiming≈1.0)")
print("bias:", net[0].bias[:3])

## 3.6 前向传播与形状检查

**调试最大技巧：在 forward 里打印形状，或写 shape 注释。**

一个典型的图像分类网络：

In [ ]:
class ImageClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),                      # 32->16
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),                      # 16->8
            nn.AdaptiveAvgPool2d(1),              # -> 1x1
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)      # (N,3,32,32) -> (N,64,1,1)
        x = self.classifier(x)    # -> (N,num_classes)
        return x

model = ImageClassifier()
x = torch.randn(8, 3, 32, 32)
print("输入:", x.shape)
print("输出:", model(x).shape)

# 逐层看形状
h = x
for i, layer in enumerate(model.features):
    h = layer(h)
    print(f"  features[{i}] {type(layer).__name__:20s} -> {tuple(h.shape)}")

# 参数量
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n总参数 {total:,}  可训练 {trainable:,}")

### 自定义 shape 打印工具（实用）

```python
def summary_shapes(model, input_shape):
    ...
```
PyTorch 没有内置 summary，需要 `pip install torchinfo` 用 `summary(model, input_size)`。
下面给一个轻量实现。

In [ ]:
model = ImageClassifier()

try:
    from torchinfo import summary
    summary(model, input_size=(8, 3, 32, 32),
            col_names=["input_size", "output_size", "num_params"])
except ImportError:
    print("未安装 torchinfo，使用内置简易统计。安装：pip install torchinfo\n")
    h = torch.randn(8, 3, 32, 32)
    for name, module in model.named_children():
        h = module(h)
        n = sum(p.numel() for p in module.parameters())
        print(f"{name:10s} -> {tuple(h.shape)}  参数 {n:,}")
    print("总计:", f"{sum(p.numel() for p in model.parameters()):,}")

> 若想看到更漂亮的逐层表格，先运行 `!pip install torchinfo`。

## 3.7 to(device)：模型搬上 GPU

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
```
之后喂给模型的数据也要 `.to(device)`。

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ImageClassifier().to(device)

x = torch.randn(4, 3, 32, 32).to(device)
with torch.no_grad():
    out = model(x)
print("模型设备:", next(model.parameters()).device)
print("输出设备:", out.device, " 形状:", out.shape)

## 3.8 练习

**练习 1**：写一个 3 层 MLP（输入 784 → 256 → 64 → 10），
包含 ReLU、Dropout，并打印参数量。

**练习 2**：写一个 `ResidualBlock`：`y = x + fc2(relu(fc1(x)))`
（要求输入输出维度相同），并测试。

**练习 3**：用 nn.Sequential 搭建同样的网络，比较 state_dict 的 key。

**练习 4**：测量模型在 train 和 eval 模式下 Dropout 输出的差异。

In [ ]:
# 练习 1
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 10),
        )
    def forward(self, x):
        return self.net(x)

m = MLP()
print("练习1 参数量:", sum(p.numel() for p in m.parameters()))
print("输出:", m(torch.randn(2, 784)).shape)

# 练习 2
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.relu = nn.ReLU()
    def forward(self, x):
        h = self.relu(self.fc1(x))
        return x + self.fc2(h)      # 残差连接

blk = ResidualBlock(32)
print("\n练习2 残差:", blk(torch.randn(4, 32)).shape)

# 练习 3
seq = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Dropout(0.2),
                    nn.Linear(256, 64), nn.ReLU(), nn.Dropout(0.2),
                    nn.Linear(64, 10))
print("练习3 Sequential keys:", list(seq.state_dict().keys()))

# 练习 4
m = MLP()
x = torch.randn(4, 784)
m.train(); a = m(x)
m.eval(); b = m(x)
print("\n练习4 train 输出含 0:", (a == 0).any().item(), " eval 输出含 0:", (b == 0).any().item())
print("train/eval 输出不同:", not torch.allclose(a, b))

## 3.9 小结

- 自定义网络 = 继承 `nn.Module` + `__init__` 定义层 + `forward` 定义流动。
- `nn.Parameter` 才会被注册为可训练参数；**动态层用 ModuleList，不要用 list**。
- 容器：`Sequential`（顺序）、`ModuleList`（列表）、`ModuleDict`（字典）。
- 训练用 `model.train()`，推理用 `model.eval()` + `torch.no_grad()`。
- 用 `torchinfo.summary` 检查形状和参数量。

下一节：`04_数据集与DataLoader.ipynb`